In [0]:
query = """
    SELECT 
        ROW_NUMBER() OVER(ORDER BY ci.customer_id) as customer_key,
        ci.customer_id,
        ci.customer_code_key,
        ci.customer_firstname,
        ci.customer_lastname,
        ci.customer_marital_status,
        CASE
            WHEN ci.customer_gender <> 'n/a' THEN ci.customer_gender
            ELSE COALESCE(ca.customer_gender,'n/a')
        END as Gender,
        la.country as customer_country,
        ca.customer_bday as customer_birthday
    FROM data_lakehouse_project.silver.silver_cust_info ci
    LEFT JOIN data_lakehouse_project.silver.silver_log_a101 la
    ON la.customer_id = ci.customer_code_key
    LEFT JOIN data_lakehouse_project.silver.silver_cust_az12 ca
    ON ca.customer_id = ci.customer_code_key
    WHERE ci.customer_id IS NOT NULL
"""
df = spark.sql(query)

In [0]:
(
    df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("data_lakehouse_project.gold.gold_dim_customer")
)

In [0]:
%sql
select * from data_lakehouse_project.gold.gold_dim_customer